# Update cellsnp output dir 

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    list(
        
        library(Seurat), 
        
        library(ggplot2), 
        library(patchwork), 
        library(RColorBrewer), 
        
        library(tidyverse), 
        library(data.table) # fread
        
    )
    
)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Source files
source("plotting_global.R")

In [ ]:
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Load vireo output

In [ ]:
# Get reference cell_id ident
so <- readRDS("data/object/pp.rds")

cell_id_ident <- data.frame(
    
    sample_name=so$sample_name, 
    cell_id_ident=sapply(strsplit(colnames(so), "_"), "[", 2)

)

rownames(cell_id_ident) <- NULL
cell_id_ident <- unique(cell_id_ident)

In [ ]:
vireo_dir <- grep("Patient_", list.dirs("data/object/genotype/vireo/vireo_mode_1a_merge_vcf/", recursive=FALSE), value=TRUE)

In [ ]:
vireo_import <- function(vireo_dir, cell_id_ident) {
    
    vireo_df <- data.table::fread(paste0(vireo_dir, "/donor_ids.tsv"))
    
    # Extract meta data from cell column
    vireo_df$cell_id <- sapply(strsplit(vireo_df$cell, "_Baseline|_Tx|_D14|_D100|_GVHD"), "[", 2)
    vireo_df$patient_id <- sapply(strsplit(vireo_df$cell, "_Baseline|_Tx|_D14|_D100|_GVHD"), "[", 1)
    vireo_df$time_point <- sapply(vireo_df$cell, function(x) c("Baseline", "Tx", "D14", "D100", "GVHD")[sapply(c("_Baseline", "_Tx", "_D14", "_D100", "_GVHD"), grepl, x)]) 
    vireo_df$sample_name <- paste0(vireo_df$patient_id, "_", vireo_df$time_point)
                                  
    # Add minMAF from vireo_dir name 
    vireo_df$minmaf <- paste0("0.", tail(strsplit(vireo_dir, "_")[[1]], n=1))
                                  
    # Add cellsnp type 
    vireo_df$db_type <- ifelse(grepl("cellsnp", vireo_dir), "cellsnp", "vartrix")
                                  
    # Add cell_id ident from Seurat object to vireo data 
    vireo_df <- dplyr::left_join(vireo_df, cell_id_ident, by="sample_name") %>% 
            dplyr::mutate(cell_id=paste0(cell_id, "_", cell_id_ident)) %>% 
            dplyr::select(-cell_id_ident)

                                  
    return(vireo_df)
                                  
}

In [ ]:
vireo <- lapply(vireo_dir, vireo_import, cell_id_ident)
vireo <- do.call("rbind", vireo)

# Mutate vireo results

In [ ]:
# Add cell type to vireo 
vireo <- dplyr::select(so@meta.data, cell_id, cell_type_leiden) %>% 
    dplyr::mutate(cell_type_domain=ifelse(.$cell_type %in% c("Mlc", "lEC", "vEC", "Pc", paste0("Fibro-", 1:5), paste0("Kc-", 1:2)), "non-immune cell", "immune cell")) %>% 
    dplyr::left_join(vireo, ., by="cell_id")

In [ ]:
# Annotate genotype for samples by time point reference 
host_id_time_point <- vireo %>% 
    dplyr::filter(patient_id %in% paste0("Patient_", 1:7)) %>% 
    dplyr::filter(time_point %in% c("Baseline", "Tx")) %>% 
    dplyr::mutate(minmaf=paste(db_type, minmaf)) %>%
    dplyr::group_by(patient_id, minmaf, donor_id) %>% 
    dplyr::summarise(n=n()) %>% 
    dplyr::filter(donor_id %in% c("donor0", "donor1")) %>% 
    dplyr::group_by(patient_id, minmaf) %>% 
    slice_max(order_by=n, n=1, with_ties=FALSE) %>% ungroup() %>% 
    dplyr::rename(host_id=donor_id) %>% dplyr::select(-n)

# Annotate genotype for samples by cell type
host_id_cell_type <- vireo %>% 
    dplyr::filter(time_point %in% c("GVHD")) %>% 
    dplyr::filter(cell_type_domain %in% c("non-immune cell")) %>% 
    dplyr::filter(patient_id %in% paste0("Patient_", 8:11)) %>%
    dplyr::mutate(minmaf=paste(db_type, minmaf)) %>%
    dplyr::group_by(patient_id, minmaf, donor_id) %>% 
    dplyr::summarise(n=n()) %>% 
    dplyr::filter(donor_id %in% c("donor0", "donor1")) %>% 
    dplyr::group_by(patient_id, minmaf) %>% 
    slice_max(order_by=n, n=1, with_ties=FALSE) %>% ungroup() %>%
    dplyr::rename(host_id=donor_id) %>% dplyr::select(-n)

host_id <- rbind(host_id_time_point, host_id_cell_type)

In [ ]:
# Add genotype to vireo result
vireo <- vireo %>% 
    dplyr::mutate(minmaf=paste(db_type, minmaf)) %>% 
    dplyr::left_join(., host_id, by=c("patient_id", "minmaf")) %>%
    dplyr::mutate(genotype_class=ifelse(.$donor_id==.$host_id, "Host", ifelse(.$donor_id %in% c("donor0", "donor1"), "Donor", .$donor_id))) %>% 
    dplyr::mutate(genotype_best_class=ifelse(.$best_singlet==.$host_id, "Host", ifelse(.$best_singlet %in% c("donor0", "donor1"), "Donor", .$best_singlet)))

In [ ]:
# XIST expression 
so[["pXIST_RNA"]] <- PercentageFeatureSet(so, features="XIST")
vireo <- dplyr::left_join(as.data.frame(vireo), so@meta.data[, c("cell_id", "pXIST_RNA")], by="cell_id")

In [ ]:
# Set factor levels 
set_factor_levels <- function(vireo) {
    
    vireo$genotype_class <- factor(vireo$genotype_class, levels=c("Host", "Donor", "unassigned", "doublet"))
    vireo$genotype_best_class <- factor(vireo$genotype_best_class, levels=c("Host", "Donor"))
    vireo$patient_id <- factor(vireo$patient_id, levels=paste0("Patient_", 1:11))
    
    return(vireo)
    
}

In [ ]:
vireo <- set_factor_levels(vireo)

# Save results 

In [ ]:
write.csv(vireo, "result/genotype/vireo_mode_1a_merge_vcf.csv")

# Vireo plotting function 

In [ ]:
vireo_plot <- function(vireo, title, patient_id_filter, time_point_filter, cell_type_domain_filter, genotype_filter) {
    
    data <- vireo %>% 
        dplyr::filter(patient_id %in% patient_id_filter) %>%
        dplyr::filter(time_point %in% time_point_filter) %>% 
        dplyr::filter(cell_type_domain %in% cell_type_domain_filter) %>% 
        dplyr::group_by(patient_id, minmaf, {{genotype_filter}}) %>% summarise(n=n()) %>% 
        dplyr::group_by(patient_id, minmaf) %>% mutate(ratio=n/sum(n))

    plot_1 <-ggplot(data, aes(x=as.character(minmaf), y=ratio, fill={{genotype_filter}})) + 
        geom_bar(stat="identity") + 
        ggtitle(title) + xlab("") + ylab("ratio") + 
        scale_fill_manual("Vireo", values=setNames(c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425"), c("Host", "Donor", "unassigned", "doublet"))) + 
        facet_grid(~patient_id) + 
        theme(axis.text.x=element_text(angle=45, vjust=1, hjust=1))
    
    data <- vireo %>% 
        dplyr::filter(patient_id %in% patient_id_filter) %>%
        dplyr::filter(time_point %in% time_point_filter) %>%
        dplyr::filter(cell_type_domain %in% cell_type_domain_filter) %>% 
        droplevels() %>% 
        tidyr::complete(patient_id, {{genotype_filter}}, minmaf, fill=list(pXIST_RNA=0), explicit=TRUE)

    plot_2 <-ggplot(data, aes(x=minmaf, y=pXIST_RNA, fill={{genotype_filter}}, color={{genotype_filter}})) + 
        geom_boxplot(position=position_dodge2(preserve="single"), outlier.size=0.5) + 
        ggtitle(title)  + xlab("") + ylab("XIST [%]") + 
        scale_color_manual("Vireo", values=setNames(c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425"), c("Host", "Donor", "unassigned", "doublet")), drop=FALSE) + 
        scale_fill_manual("Vireo", values=setNames(c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425"), c("Host", "Donor", "unassigned", "doublet")), drop=FALSE) + 
        facet_grid(~patient_id,) + 
        theme(axis.text.x=element_text(angle=45, vjust=1, hjust=1))
    
    data <- vireo %>% 
        dplyr::filter(patient_id %in% patient_id_filter) %>%
        dplyr::filter(time_point %in% time_point_filter) %>% 
        dplyr::filter(cell_type_domain %in% cell_type_domain_filter)

    plot_3 <- ggplot(data, aes(x=prob_max, ..scaled.., color=minmaf, color=NULL)) + 
        geom_density(bins=20) + 
        ggtitle(title)  + xlab("") + ylab("Scaled density") + 
        scale_color_manual(values=circlize::colorRamp2(4:9, brewer.pal(9,"Blues")[4:9])(1:10)) +
        facet_grid(~patient_id)
    
    plot <- plot_1 + plot_2 + plot_3 + plot_layout(nrow=3)
    
    return(plot)
    
}

# Genotype all CellRanger raw cells 

## Raw cells (Donor ID)

In [ ]:
options(repr.plot.width=11*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "All raw cells", paste0("Patient_", 1:11), c("Baseline", "Tx", "D14", "D100", "GVHD"), c("immune cell", "non-immune cell"), genotype_class)

## Raw cells (Best singlet)

In [ ]:
options(repr.plot.width=11*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "All raw cells", paste0("Patient_", 1:11), c("Baseline", "Tx", "D14", "D100", "GVHD"), c("immune cell", "non-immune cell"), genotype_best_class)

# Filter vireo result by cell id

In [ ]:
vireo <- vireo[vireo$cell_id %in% so$cell_id, ]
vireo <- set_factor_levels(vireo)

# Genotype distribution by time point 

## Baseline/Tx samples (Donor ID)

In [ ]:
options(repr.plot.width=6*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "Basline/Tx samples", paste0("Patient_", c(1, 3:7)), c("Baseline", "Tx"), c("immune cell", "non-immune cell"), genotype_class)

## D14/D100 samples (Donor ID)

In [ ]:
options(repr.plot.width=6*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "D14/D100 samples", paste0("Patient_", c(1, 3:7)), c("D14", "D100"), c("immune cell", "non-immune cell"), genotype_class)

## Baseline/Tx samples (Best singlet)

In [ ]:
options(repr.plot.width=6*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "Basline/Tx samples", paste0("Patient_", c(1, 3:7)), c("Baseline", "Tx"), c("immune cell", "non-immune cell"), genotype_best_class)

## D14/D100 samples (Best singlet)

In [ ]:
options(repr.plot.width=6*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "D14/D100 samples", paste0("Patient_", c(1, 3:7)), c("D14", "D100"), c("immune cell", "non-immune cell"), genotype_best_class)

# Genotype distribution by cell type 

## Non-immune cells samples (Donor ID)

In [ ]:
options(repr.plot.width=4*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "Non-immune cells", paste0("Patient_", 8:11), c("GVHD"), c("non-immune cell"), genotype_class)

## Immune cells samples (Donor ID)

In [ ]:
options(repr.plot.width=4*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "Immune cells", paste0("Patient_", 8:11), c("GVHD"), c("immune cell"), genotype_class)

## Non-immune cells samples (Best singlet)

In [ ]:
options(repr.plot.width=4*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "Non-immune cells", paste0("Patient_", 8:11), c("GVHD"), c("non-immune cell"), genotype_best_class)

## Immune cells samples (Best singlet)

In [ ]:
options(repr.plot.width=4*3.5+2, repr.plot.height=15)

vireo_plot(vireo, "Immune cells", paste0("Patient_", 8:11), c("GVHD"), c("immune cell"), genotype_best_class)